# Walmart : predict weekly sales
This notebook demonstrates exploratory data analysis (EDA), preprocessing, unsupervised learning (PCA & K-Means), and supervised learning (Linear, Ridge, Lasso) to predict store weekly sales.

## 1. Import libraries


In [ ]:
lang = 'en'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')


## 2. Load dataset and Exploratory Data Analysis (EDA)
Let's load the dataset and display the first rows and general statistics.


In [ ]:
# Load the dataset
df = pd.read_csv('Walmart_Store_sales.csv')
display(df.head())
print("Shape of dataset:", df.shape)


In [ ]:
# Basic statistics
display(df.describe(include='all'))
print("Missing values:\n", df.isnull().sum())


### Visualizing missing values
We can plot the missing values to get a better sense of data completion.


In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing values heatmap' if lang == 'en' else 'Carte de chaleur des valeurs manquantes')
plt.show()


### Distribution of target variable (`Weekly_Sales`)


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Weekly_Sales'], bins=50, kde=True)
plt.title('Distribution of Weekly Sales' if lang == 'en' else 'Distribution des ventes hebdomadaires')
plt.xlabel('Weekly Sales')
plt.ylabel('Frequency' if lang == 'en' else 'Fréquence')
plt.show()


## 3. Preprocessing (Pandas)
- Drop rows where `Weekly_Sales` is missing.
- Create `Year`, `Month`, `Day`, and `DayOfWeek` from `Date`.
- Drop rows containing outliers for numerical variables.


In [ ]:
# Drop missing target values
df = df.dropna(subset=['Weekly_Sales'])

# Create datetime features
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df = df.drop('Date', axis=1)

# Remove outliers ([mean - 3*std, mean + 3*std])
num_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
for col in num_features:
    mean = df[col].mean()
    std = df[col].std()
    df = df[(df[col] >= mean - 3*std) & (df[col] <= mean + 3*std) | df[col].isnull()]

print("Shape after pandas preprocessing:", df.shape)


## 4. Unsupervised Machine Learning
### Principal Component Analysis (PCA)


In [ ]:
# Prepare data for PCA (only numerical features)
X_num = df[num_features].dropna()
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_num)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, c='b')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
plt.title("PCA - 2 Components" if lang == 'en' else "PCA - 2 Composantes")
plt.show()


### K-Means Clustering
Let's apply K-Means to find potential groups within the data.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters (k=3)" if lang == 'en' else "Clusters K-Means (k=3)")
plt.show()


## 5. Supervised Machine Learning - Preprocessing pipeline
Prepare Scikit-Learn pipelines to handle missing values and scaling/encoding.


In [ ]:
# Separate target and features
target_name = 'Weekly_Sales'
Y = df[target_name]
X = df.drop(target_name, axis=1)

# Split into train/test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define pipelines
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'DayOfWeek']
categorical_features = ['Store', 'Holiday_Flag']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


## 6. Baseline Model: Linear Regression
Train and evaluate a standard Linear Regression model.


In [ ]:
# Train Baseline
baseline = LinearRegression()
baseline.fit(X_train_processed, Y_train)

# Predictions
Y_train_pred = baseline.predict(X_train_processed)
Y_test_pred = baseline.predict(X_test_processed)

# Evaluation
print("--- Baseline Model ---")
print("Train R2 Score:", r2_score(Y_train, Y_train_pred))
print("Test R2 Score:", r2_score(Y_test, Y_test_pred))

# Get feature importance
feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))
coefs = pd.DataFrame({'Feature': feature_names, 'Coefficient': baseline.coef_}).sort_values(by='Coefficient', ascending=False)
display(coefs.head())
display(coefs.tail())


## 7. Regularized Models: Ridge & Lasso
Use GridSearchCV to find optimal regularization strength.


In [ ]:
# GridSearchCV for Ridge
ridge = Ridge()
params_ridge = {'alpha': [0.01, 0.1, 1, 10, 100, 1000]}
grid_ridge = GridSearchCV(ridge, params_ridge, cv=5, scoring='r2')
grid_ridge.fit(X_train_processed, Y_train)

print("Best alpha for Ridge:", grid_ridge.best_params_['alpha'])
print("Train R2 (Ridge):", r2_score(Y_train, grid_ridge.predict(X_train_processed)))
print("Test R2 (Ridge):", r2_score(Y_test, grid_ridge.predict(X_test_processed)))


In [ ]:
# GridSearchCV for Lasso
lasso = Lasso(max_iter=10000)
params_lasso = {'alpha': [1, 10, 100, 500, 1000, 2000]}
grid_lasso = GridSearchCV(lasso, params_lasso, cv=5, scoring='r2')
grid_lasso.fit(X_train_processed, Y_train)

print("Best alpha for Lasso:", grid_lasso.best_params_['alpha'])
print("Train R2 (Lasso):", r2_score(Y_train, grid_lasso.predict(X_train_processed)))
print("Test R2 (Lasso):", r2_score(Y_test, grid_lasso.predict(X_test_processed)))
